# Day 4 — 대시보드 통합

Day 1~3에서 만든 모든 기능을 **하나의 HTML 파일**로 통합합니다.

## 대시보드 구성
```
┌──────────────────────────────────────────────────┐
│  [섹션 1] 시세 현황                               │
│  현재가 카드 4개 + 볼린저 밴드 2×2 차트            │
├──────────────────────────────────────────────────┤
│  [섹션 2] 포트폴리오                              │
│  자산/수익률/MDD 카드 + 자산 곡선 + 히트맵         │
├──────────────────────────────────────────────────┤
│  [섹션 3] 백테스팅                               │
│  성과 요약 테이블 + 매매 마커 + 자산 곡선          │
└──────────────────────────────────────────────────┘
```

In [ ]:
from datetime import datetime

%%capture
%run 01_data_pipeline.ipynb

In [ ]:
%%capture
%run 02_portfolio.ipynb

In [ ]:
%%capture
%run 03_backtesting.ipynb

## 1. 섹션별 HTML 조각 생성

### 메트릭 카드 만들기
수치 정보를 보기 좋은 카드 형태의 HTML로 만듭니다.

In [ ]:
def make_card(label, value, color="black"):
    """
    지표 카드 HTML 조각을 반환합니다.
    label: 카드 제목 (예: 'KRW-BTC')
    value: 카드 수치 (예: '119,004,000 원')
    color: 수치 글자 색 (양수 red, 음수 blue, 기본 black)
    """
    return f"""
    <div class="card">
        <div class="card-label">{label}</div>
        <div class="card-value" style="color:{color}">{value}</div>
    </div>"""

def make_cards_row(cards_html):
    """카드들을 가로로 나열하는 div로 감쌉니다."""
    return f'<div class="cards">{" ".join(cards_html)}</div>'

In [ ]:
def make_section1_html(data, fig_bollinger):
    """
    섹션 1: 시세 현황
    현재가 카드 4개 + 볼린저 밴드 2×2 차트
    """
    cards = []
    for ticker in data:
        last   = data[ticker].iloc[-1]
        price  = format_price(last["close"])
        ret_7d = last["return_7d"]
        color  = "red" if ret_7d >= 0 else "blue"
        ret_str = format_pct(ret_7d)
        cards.append(make_card(
            f"{ticker}<br><small style='font-size:11px;color:#888'>{ret_str} (7d)</small>",
            price, color
        ))

    chart_html = fig_bollinger.to_html(full_html=False, include_plotlyjs="cdn")

    return f"""
    <div class="section">
        <h2>📈 시세 현황</h2>
        {make_cards_row(cards)}
        {chart_html}
    </div>"""


In [ ]:
def make_section2_html(portfolio_values, port_data, fig_portfolio, fig_heatmap_html):
    """
    섹션 2: 포트폴리오
    총 자산 / 수익률 / MDD 카드 + 자산 곡선 + 히트맵
    """
    start_val = portfolio_values.iloc[0]
    end_val   = portfolio_values.iloc[-1]
    total_ret = (end_val - start_val) / start_val * 100
    mdd       = calc_mdd(portfolio_values)

    ret_color = "red" if total_ret >= 0 else "blue"
    mdd_color = "blue"

    cards = [
        make_card("현재 자산",  f"{end_val:,.0f} 원"),
        make_card("총 수익률",  f"{total_ret:+.2f}%", ret_color),
        make_card("MDD",       f"{mdd:.2f}%", mdd_color),
    ]

    chart_html = fig_portfolio.to_html(full_html=False, include_plotlyjs=False)

    return f"""
    <div class="section">
        <h2>💼 포트폴리오 분석</h2>
        {make_cards_row(cards)}
        {chart_html}
        {fig_heatmap_html}
    </div>"""


In [ ]:
def make_section3_html(stats, bh_return, fig_backtest):
    """
    섹션 3: 백테스팅
    성과 요약 테이블 + 매매 마커/자산 곡선 차트
    """
    excess = stats["total_ret"] - bh_return

    table_rows = [
        ("총 수익률",       f"{stats['total_ret']:+.2f}%"),
        ("MDD",            f"{stats['mdd']:.2f}%"),
        ("승률",           f"{stats['win_rate']:.1f}%"),
        ("평균 수익 거래",  f"{stats['avg_win']:+.2f}%"),
        ("평균 손실 거래",  f"{stats['avg_loss']:+.2f}%"),
        ("Buy & Hold",     f"{bh_return:+.2f}%"),
        ("전략 초과 수익",  f"{excess:+.2f}%p"),
    ]
    rows_html = "".join(
        f"<tr><td>{k}</td><td><b>{v}</b></td></tr>" for k, v in table_rows
    )
    table_html = f"""
    <table style="border-collapse:collapse;margin-bottom:16px;min-width:300px">
        <thead><tr style="background:#f0f0f0">
            <th style="padding:8px 16px;text-align:left">항목</th>
            <th style="padding:8px 16px;text-align:left">값</th>
        </tr></thead>
        <tbody>{rows_html}</tbody>
    </table>"""

    chart_html = fig_backtest.to_html(full_html=False, include_plotlyjs=False)

    return f"""
    <div class="section">
        <h2>🔬 백테스팅 (골든크로스 전략 / KRW-BTC)</h2>
        {table_html}
        {chart_html}
    </div>"""


## 2. 히트맵 → HTML 변환

seaborn 히트맵은 matplotlib 이미지이므로 base64로 인코딩해서 HTML에 삽입합니다.

In [ ]:
import base64
from io import BytesIO

def heatmap_to_html(port_data):
    """seaborn 히트맵을 base64 이미지 HTML로 변환합니다."""
    returns_df = pd.DataFrame({
        ticker: port_data[ticker]["close"].pct_change() * 100
        for ticker in port_data
    }).dropna()
    corr = returns_df.corr()

    fig_hm, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
                vmin=-1, vmax=1, ax=ax)
    ax.set_title("4종목 일별 수익률 상관관계")
    plt.tight_layout()

    buf = BytesIO()
    fig_hm.savefig(buf, format="png", dpi=100)
    plt.close(fig_hm)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode()
    return f'<img src="data:image/png;base64,{img_b64}" style="max-width:600px;margin:12px 0">'


## 3. 대시보드 전체 조립

In [ ]:
def build_dashboard():
    """
    Day 1~3 함수를 재사용해서 대시보드 HTML을 조립합니다.
    Day 1~3 함수를 수정 없이 그대로 사용합니다.
    """
    now = datetime.now().strftime("%Y-%m-%d %H:%M")

    # ── 차트 생성 ─────────────────────────────────────────────────
    fig_bollinger = make_bollinger_chart(data)       # Day 1 함수

    port_data_d4        = get_portfolio_data(data, START_DAYS_AGO)  # Day 2 함수
    holdings_d4         = calc_holdings(port_data_d4, PORTFOLIO, FEE_RATE)
    portfolio_values_d4 = calc_daily_values(port_data_d4, holdings_d4)
    fig_portfolio       = make_portfolio_chart(portfolio_values_d4, port_data_d4)  # Day 2 함수
    heatmap_html        = heatmap_to_html(port_data_d4)

    df_sig_d4  = generate_signals(df_btc)            # Day 3 함수
    trades_d4, final_cap_d4 = simulate_trades(
        df_sig_d4, BACKTEST_CONFIG["capital"], BACKTEST_CONFIG["fee_rate"]
    )
    bh_ret_d4  = calc_bh_return(df_btc, BACKTEST_CONFIG["capital"], BACKTEST_CONFIG["fee_rate"])
    stats_d4   = calc_trade_stats(trades_d4, BACKTEST_CONFIG["capital"], final_cap_d4)
    strat_curve, bh_curve_d4 = build_equity_curves(
        df_btc, trades_d4, BACKTEST_CONFIG["capital"], BACKTEST_CONFIG["fee_rate"]
    )
    fig_backtest = make_backtest_chart(df_sig_d4, trades_d4, strat_curve, bh_curve_d4)

    # ── 섹션별 HTML 생성 ──────────────────────────────────────────
    s1 = make_section1_html(data, fig_bollinger)
    s2 = make_section2_html(portfolio_values_d4, port_data_d4, fig_portfolio, heatmap_html)
    s3 = make_section3_html(stats_d4, bh_ret_d4, fig_backtest)

    # ── HTML 전체 조립 ────────────────────────────────────────────
    css = """
        body  { font-family: Arial, sans-serif; background:#f5f5f5; padding:24px; margin:0; }
        h1    { color:#222; margin-bottom:4px; }
        .section { background:white; border-radius:12px; padding:24px;
                   margin-bottom:20px; box-shadow:0 2px 8px rgba(0,0,0,0.06); }
        .section h2 { margin-top:0; color:#333; }
        .cards { display:flex; gap:12px; flex-wrap:wrap; margin-bottom:16px; }
        .card  { background:#f8f9fa; border-radius:8px; padding:16px 24px;
                 text-align:center; min-width:150px; }
        .card-label { font-size:12px; color:#888; margin-bottom:6px; }
        .card-value { font-size:20px; font-weight:700; }
        table td, table th { padding:8px 16px; border-bottom:1px solid #eee; }
    """

    html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <title>암호화폐 포트폴리오 대시보드</title>
  <style>{css}</style>
</head>
<body>
  <h1>🪙 암호화폐 포트폴리오 대시보드</h1>
  <p style="color:#888;margin-top:0">생성: {now}</p>
  {s1}
  {s2}
  {s3}
</body>
</html>"""
    return html


In [ ]:
# ── 대시보드 생성 & 저장 ─────────────────────────────────────────
import os

html_content = build_dashboard()

output_path = os.path.join(os.getcwd(), "dashboard.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_content)

file_size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"저장 완료: {output_path}")
print(f"파일 크기: {file_size_mb:.2f} MB  ({'✅ 10MB 이하' if file_size_mb < 10 else '⚠️ 10MB 초과'})")

In [ ]:
# ── 노트북 안에서 미리보기 ───────────────────────────────────────
from IPython.display import IFrame
IFrame("dashboard.html", width="100%", height=1400)